# 01. Извлечение данных из Hive

> **Краткое резюме для руководителя**: Этот ноутбук извлекает данные из банковских таблиц Hive. Начиная с одной компании (seed), он расширяет окружение через транзакционные связи (hop-by-hop), собирая информацию о контрагентах, транзакциях, доверенностях и зарплатных проектах. Также извлекаются ОКВЭД-коды (отрасль) и коды регионов для каждого клиента. Результат — набор Parquet-файлов, готовых для построения графа.

**User Story 1**: Как аналитик, я хочу извлечь N-hop окружение seed-компании из Hive, чтобы получить данные для построения графа связей.

**Процесс**:
1. Задаём seed-компанию (client_uk) и параметры (период, количество хопов)
2. Расширяем окружение через транзакции (hop by hop) с контролем размера
3. Извлекаем узлы с атрибутами (тип, ИНН, ОКВЭД, регион)
4. Извлекаем транзакционные, доверенные и зарплатные рёбра
5. Сохраняем в Parquet-файлы

**Предпосылки**: Запустите `00_verify_schema.ipynb` для проверки колонок.

---

## Параметры

Настройте параметры перед запуском:
- **SEED_CLIENT_UK** — client_uk компании, вокруг которой строится граф
- **N_HOPS** — глубина расширения (1-2 рекомендуется; 3+ = очень большой граф)
- **MIN_TX_COUNT_HOP** — минимум транзакций для включения контрагента при расширении хопа (↑ = меньше узлов, быстрее)
- **MAX_NEIGHBORHOOD_SIZE** — жёсткий лимит: если размер окружения превышает порог, расширение останавливается

In [ ]:
# =====================================================
# НАСТРОЙТЕ ЭТИ ПАРАМЕТРЫ ПЕРЕД ЗАПУСКОМ
# =====================================================

SEED_CLIENT_UK = 12345678      # Замените на реальный client_uk (CAST uk AS BIGINT)
N_HOPS = 2                      # Количество хопов (рекомендуется 1-2)
START_DATE = '2025-01-01'       # Начало периода
END_DATE = '2025-12-31'         # Конец периода

# Минимум транзакций для включения контрагента в хоп.
# Увеличьте если окружение растёт слишком сильно (>10K клиентов).
# Например: 3 = включать только тех, с кем было ≥3 транзакций.
MIN_TX_COUNT_HOP = 3

# Жёсткий лимит размера окружения.
# Если после хопа превышен — расширение останавливается с предупреждением.
MAX_NEIGHBORHOOD_SIZE = 10_000

# Директория для Parquet-файлов (абсолютный путь на локальной ФС)
import os
OUTPUT_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

## Инициализация

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

from src import config
from src.etl import extract_seed_neighborhood

# На MDP SparkSession уже инициализирована платформой
try:
    spark
    print("SparkSession already available (MDP)")
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .enableHiveSupport() \
        .getOrCreate()
    print("SparkSession created")

print(f"Spark version: {spark.version}")
print(f"Database: {config.HIVE_DATABASE}")
print(f"Seed: {SEED_CLIENT_UK}, Hops: {N_HOPS}, Period: {START_DATE} — {END_DATE}")

## Извлечение окружения

Функция `extract_seed_neighborhood` выполняет полный цикл ETL:

1. **Hop-расширение**: начиная с seed, находит контрагентов через транзакции (таблица `buh_sbp_oper`), затем контрагентов контрагентов (2-й хоп), и т.д. На каждом шаге фильтрует по MIN_TX_COUNT_HOP
2. **Извлечение узлов**: из `client_sdim` — атрибуты клиентов (тип, имя, ИНН, **ОКВЭД-код**, **код региона**)
3. **Транзакционные рёбра**: суммы, количество, даты первой/последней транзакции
4. **Доверенности**: связи ЮЛ → ФЛ (таблица `poa_data`)
5. **Зарплаты**: связи ЮЛ → ФЛ (таблица `salary_data`)
6. **Hop distances**: расстояние каждого узла от seed (0 = сам seed, 1 = прямой контрагент, ...)

In [ ]:
paths = extract_seed_neighborhood(
    spark=spark,
    seed_client_uk=SEED_CLIENT_UK,
    n_hops=N_HOPS,
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=OUTPUT_DIR,
    min_tx_count_hop=MIN_TX_COUNT_HOP,
    max_neighborhood_size=MAX_NEIGHBORHOOD_SIZE,
)

print("\nOutput files:")
for name, path in paths.items():
    print(f"  {name}: {path}")

## Сводная статистика

Проверяем результат извлечения: количество узлов, рёбер разных типов, распределение по типам клиентов и расстоянию от seed.

**Ожидаемые выходные файлы**:
- `nodes.parquet` — узлы с полями: client_uk, client_type_name, name, inn, okved_code, region_code
- `transaction_edges.parquet` — транзакционные рёбра
- `authority_edges.parquet` — доверенности
- `salary_edges.parquet` — зарплаты
- `hop_distances.parquet` — расстояния от seed

In [ ]:
import pandas as pd

# Load and show summary — используем file:// для Spark read с локальной ФС
spark_prefix = f'file://{os.path.abspath(OUTPUT_DIR)}'

nodes_df = spark.read.parquet(f'{spark_prefix}/nodes.parquet')
tx_df = spark.read.parquet(f'{spark_prefix}/transaction_edges.parquet')
auth_df = spark.read.parquet(f'{spark_prefix}/authority_edges.parquet')
sal_df = spark.read.parquet(f'{spark_prefix}/salary_edges.parquet')

print("=" * 50)
print("EXTRACTION SUMMARY")
print("=" * 50)
print(f"Nodes:              {nodes_df.count():,}")
print(f"Transaction edges:  {tx_df.count():,}")
print(f"Authority edges:    {auth_df.count():,}")
print(f"Salary edges:       {sal_df.count():,}")

In [ ]:
# Node distribution by type
print("\nNodes by type:")
nodes_df.groupBy('client_type_name').count().orderBy('count', ascending=False).show(truncate=False)

In [ ]:
# hop_distances сохранён через Pandas (одиночный файл, не Spark-директория)
import pandas as pd
hop_pdf = pd.read_parquet(f'{OUTPUT_DIR}/hop_distances.parquet')
print("Nodes by hop distance:")
print(hop_pdf.groupby('hop_distance').size().reset_index(name='count').to_string(index=False))

In [ ]:
# Sample data previews
print("\n--- Nodes sample ---")
nodes_df.show(5, truncate=40)

print("\n--- Transaction edges sample ---")
tx_df.show(5, truncate=30)

print("\n--- Authority edges sample ---")
auth_df.show(5, truncate=30)

print("\n--- Salary edges sample ---")
sal_df.show(5, truncate=30)

---

## Глоссарий терминов

| Термин | Описание |
|--------|----------|
| **Seed-компания** | Компания, с которой начинается построение графа; все остальные узлы — её контрагенты или контрагенты контрагентов |
| **Hop (хоп)** | Один шаг расширения: hop 0 = seed, hop 1 = прямые контрагенты seed, hop 2 = контрагенты контрагентов |
| **client_uk** | Уникальный ключ клиента в банковской системе (BIGINT) |
| **client_type_name** | Тип клиента: «company» (ЮЛ), «individual» (ФЛ), «sole_proprietor» (ИП) |
| **ОКВЭД-код (okved_code)** | Двузначный код отрасли по Общероссийскому классификатору видов экономической деятельности (напр. 46 = оптовая торговля). Для физлиц = «00» |
| **Код региона (region_code)** | Двузначный код региона регистрации (напр. 77 = Москва, 78 = Санкт-Петербург). Для физлиц = «00» |
| **Parquet** | Колоночный формат хранения данных; эффективен для аналитических запросов и экономит место |
| **MIN_TX_COUNT_HOP** | Фильтр при расширении: включает контрагента в следующий хоп только если с ним ≥N транзакций |
| **MAX_NEIGHBORHOOD_SIZE** | Предохранитель: если окружение станет слишком большим, расширение останавливается |

---

**Следующий шаг**: Откройте `02_graph_construction.ipynb` для построения графа из извлечённых данных.